# Understand any stock’s risk (AOM quickstart)

**Run cells top to bottom.**  
1. Install the SDK (`riskmodels-py`).  
2. Provide your **API key** (Colab Secrets or paste). Get one at [riskmodels.app/get-key](https://riskmodels.app/get-key).  
3. Run the four examples: TSLA attribution, comparison, exposure snapshot, exposure→hedge chain.

Package import name is **`riskmodels`** (PyPI: `riskmodels-py`).

In [ ]:
# AOM (`rm`/`run`) ships in 0.3.1+ — not on PyPI until published.
# Prefer PyPI after release: %pip install -q "riskmodels-py>=0.3.1,<0.4"
%pip install -q "git+https://github.com/BlueWaterCorp/RiskModels_API.git@main#subdirectory=sdk"

## API key

**Google Colab:** click the **key icon** → **Secrets** → add **`RISKMODELS_API_KEY`** (same name as the env var).  
**Or:** leave Secrets empty and paste when the next cell prompts (input is hidden where supported).

**Local Jupyter:** export `RISKMODELS_API_KEY` before launching, or paste in the next cell.

In [ ]:
import os


def ensure_riskmodels_api_key() -> None:
    existing = (os.environ.get("RISKMODELS_API_KEY") or "").strip()
    if existing:
        print("Using RISKMODELS_API_KEY from the environment.")
        return
    try:
        from google.colab import userdata

        secret = userdata.get("RISKMODELS_API_KEY")
        if secret:
            os.environ["RISKMODELS_API_KEY"] = str(secret).strip()
            print("Loaded RISKMODELS_API_KEY from Colab Secrets.")
            return
    except ImportError:
        pass
    except Exception as e:
        print("Colab Secrets not available:", e)
    from getpass import getpass

    os.environ["RISKMODELS_API_KEY"] = getpass("Paste RISKMODELS_API_KEY (hidden where supported): ").strip()
    if not os.environ["RISKMODELS_API_KEY"]:
        raise RuntimeError("RISKMODELS_API_KEY is required. Get a key at https://riskmodels.app/get-key")


ensure_riskmodels_api_key()

## 1. TSLA — “why did it move?” (return attribution)

In [ ]:
from riskmodels import RiskModelsClient, rm, run
from riskmodels.aom import stock

client = RiskModelsClient.from_env()

req = (
    rm()
    .subject(stock("TSLA"))
    .scope(date_range_preset="ytd", as_of="latest")
    .return_attribution(resolution="full_stack", view="timeseries")
    .explain()
)

out = run(client, req)
out

## 2. Comparison — AAPL vs NVDA

In [ ]:
from riskmodels.aom import comparison

req = (
    rm()
    .subject(comparison([stock("AAPL"), stock("NVDA")]))
    .scope(date_range_preset="ytd")
    .return_attribution(resolution="full_stack", view="timeseries")
    .structured()
)

run(client, req)

## 3. Exposure snapshot

In [ ]:
req = (
    rm()
    .subject(stock("NVDA"))
    .scope(as_of="latest")
    .exposure(resolution="full_stack", view="snapshot")
    .structured()
)

run(client, req)

## 4. Exposure → hedge chain

In [ ]:
from riskmodels.aom import analyze, hedge_action

req = (
    rm()
    .subject(stock("NVDA"))
    .scope(as_of="latest")
    .chain(
        analyze(lens="exposure", resolution="full_stack", view="snapshot"),
        hedge_action(),
    )
    .structured()
)

run(client, req)